# Taxi Trip Forecasting

## Project Overview

Forecasts **daily taxi trip count** over a 14-day horizon. Synthetic data spans 2 years with weekly commuter patterns, seasonal tourism effects, and holiday spikes.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily taxi trip counts, predict the next 14 days. Taxi fleet operators need demand forecasts for fleet sizing, driver scheduling, and revenue optimization.

## Why This Project Matters

Taxi operations require balancing supply (available cabs) with demand (trip requests). Under-supply means lost revenue; over-supply means idle vehicles. Short-term forecasting enables efficient fleet management and maximizes utilization.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily taxi trip counts
- Weekly pattern (weekday commuter peaks, weekend leisure)
- Seasonal tourism effect (summer boost)
- Holiday spikes (airport travel, events)
- Slight decline trend (ride-hailing competition)

| Column | Description |
|--------|-------------|
| `date` | Date |
| `trips` | Daily taxi trip count |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'trips'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

base = 25000 - 2 * t  # decline from ride-hailing competition
weekly = np.zeros(N_POINTS)
for i in range(N_POINTS):
    dow = dates[i].dayofweek
    if dow <= 4: weekly[i] = 3000  # weekday commuters
    elif dow == 5: weekly[i] = 1000  # Saturday leisure
    else: weekly[i] = -2000  # Sunday lowest

# Summer tourism boost
seasonal = 2000 * np.sin(2 * np.pi * (t - 100) / 365)

# Holiday travel spikes (airport, family visits)
holiday = np.zeros(N_POINTS)
for i in range(N_POINTS):
    m, d = dates[i].month, dates[i].day
    if m == 11 and 22 <= d <= 28: holiday[i] = 3000  # Thanksgiving travel
    elif (m == 12 and d >= 20) or (m == 1 and d <= 3): holiday[i] = 4000  # Christmas/NY
    elif m == 7 and 3 <= d <= 5: holiday[i] = 2000  # July 4th

noise = np.random.normal(0, 1000, N_POINTS)
trips = base + weekly + seasonal + holiday + noise
trips = np.maximum(trips, 5000).round(0).astype(int)

df = pd.DataFrame({'date': dates, 'trips': trips})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,trips
0,2022-01-01,28519
1,2022-01-02,24878
2,2022-01-03,30657
3,2022-01-04,27527
4,2022-01-05,25765


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('trips Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("trips Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

trips Statistics:
count      730.000000
mean     26488.473973
std       2473.334485
min      17776.000000
25%      24880.250000
50%      26762.500000
75%      28306.250000
max      33342.000000
Name: trips, dtype: float64

CV: 0.093
Skewness: -0.481


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:     5787.4 | RMSE:     6258.6 | MAPE: 20.90%
Seasonal Naive (7)             | MAE:     1969.3 | RMSE:     2595.1 | MAPE:  7.16%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                           Adjusted R-Squared   R-Squared          RMSE  Time Taken
Model                                                                              
KernelRidge                       2026.297334 -154.792103  26303.689745    0.043807
MLPRegressor                      1608.907326 -122.685179  23437.041273    0.515460
LinearSVR                         1567.832629 -119.525587  23135.749925    0.017292
GaussianProcessRegressor          1207.329817  -91.794601  20300.433358    0.091366
RANSACRegressor                     56.254355   -3.250335   4344.657371    0.106313
QuantileRegressor                   43.590588   -2.276199   3814.425121    0.065851
SVR                                 42.695568   -2.207351   3774.133196    0.039323
NuSVR                               39.565602   -1.966585   3629.713395    0.049518
DummyRegressor                      38.217459   -1.862881   3565.706888    0.011389
OrthogonalMatchingPursuit           15.824110   -0.1403

## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: lgbm
FLAML (lgbm)                   | MAE:     1021.9 | RMSE:     1331.2 | MAPE:  4.59%
FLAML Test (lgbm)              | MAE:     1045.8 | RMSE:     1222.1 | MAPE:  4.00%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:     2673.4 | RMSE:     2966.8 | MAPE:  9.73%
SF AutoETS                     | MAE:     2839.7 | RMSE:     3118.3 | MAPE: 10.35%
SF SeasonalNaive               | MAE:     2935.3 | RMSE:     3173.0 | MAPE: 10.69%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
             model         MAE        RMSE      MAPE
      FLAML (lgbm) 1021.871294 1331.248214  4.593877
 FLAML Test (lgbm) 1045.801199 1222.147718  4.004512
Seasonal Naive (7) 1969.285714 2595.136082  7.159618
      SF AutoARIMA 2673.439941 2966.820015  9.725271
        SF AutoETS 2839.654541 3118.284945 10.354966
  SF SeasonalNaive 2935.285645 3173.048534 10.690174
Naive (Last Value) 5787.428571 6258.559134 20.896354

Top 3:
             model         MAE        RMSE     MAPE
      FLAML (lgbm) 1021.871294 1331.248214 4.593877
 FLAML Test (lgbm) 1045.801199 1222.147718 4.004512
Seasonal Naive (7) 1969.285714 2595.136082 7.159618


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: -165.24, Std: 1210.93


## Interpretation and Business Insight

- Taxi trips show strong weekday peaks (commuter-driven)
- Sunday is consistently the lowest demand day
- Holiday periods create spikes (airport travel)
- Declining trend reflects ride-hailing competition
- Summer tourism adds seasonal lift
- Weather and events are key missing predictors

## Limitations

1. Synthetic — real taxi data depends on weather, events, ride-hailing prices
2. No weather features (rain dramatically increases taxi demand)
3. No airport vs city center segmentation
4. Daily granularity — rush hours matter most
5. No competitor pricing data

## How to Improve This Project

1. Add weather data (rain/snow → taxi demand surge)
2. Use hourly data for shift planning
3. Segment by zone (airport, downtown, residential)
4. Add ride-hailing competitor pricing
5. Include flight arrival data for airport queue prediction

## Production Considerations

- Hourly demand forecasting for fleet dispatch
- Airport queue management
- Driver shift optimization
- Revenue-per-trip forecasting

## Common Mistakes

1. Ignoring the declining trend (ride-hailing competition)
2. Not handling weather effects (rain → 30%+ demand increase)
3. Using daily totals for hourly dispatch decisions
4. Treating airport and city trips as the same market

## Mini Challenge / Exercises

1. Add a synthetic rain feature and measure demand elasticity
2. Model the declining trend — when does fleet right-sizing matter?
3. Separate airport vs non-airport demand patterns
4. Build a peak-hour demand predictor using hourly mock data

## Final Summary / Key Takeaways

1. Taxi trip forecasting has strong weekly patterns driven by commuting
2. Holiday periods create significant spikes
3. Ride-hailing competition creates a structural decline
4. Weather is the dominant short-term driver
5. Zone-level and hourly forecasts are needed for real operations

In [18]:
import json
metrics = {
    'project': 'Taxi Trip Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Taxi Trip Forecasting — Complete ===")

Metrics saved.

=== Taxi Trip Forecasting — Complete ===
